# Seed Viggle-Animate small weights

Downloads LoRA + text_cond + VAE (~4.2GB) and publishes to
`marsfmmg/viggle-animate-weights`. Run this first.


In [ ]:
# === Download small weight files ===
import subprocess, os

SEED = "/kaggle/working/seed"
FILES = [
    ("loras/viggle_animate_dmd_lora_r64.safetensors", 941893032,
     "https://huggingface.co/drbaph/Viggle-Animate-ComfyUI/resolve/main/loras/viggle_animate_dmd_lora_r64.safetensors"),
    ("text_cond/fixed_embed_fwd_anyframe.safetensors", 3713456,
     "https://huggingface.co/drbaph/Viggle-Animate-ComfyUI/resolve/main/text_cond/fixed_embed_fwd_anyframe.safetensors"),
    ("vae/minimax_h3_video_vae_int8_convrot.safetensors", 3171670912,
     "https://huggingface.co/Kijai/MiniMax-H3-experimental/resolve/main/minimax_h3_video_vae_int8_convrot.safetensors")
]

for relpath, expected, url in FILES:
    out = os.path.join(SEED, relpath)
    os.makedirs(os.path.dirname(out), exist_ok=True)
    if os.path.exists(out) and os.path.getsize(out) == expected:
        print(f"SKIP {relpath} (already present)")
        continue
    print(f"Downloading {relpath} ({expected/1e9:.2f} GB)...")
    subprocess.run(["wget", "-q", "--show-progress", url, "-O", out], check=True)
    actual = os.path.getsize(out)
    assert actual == expected, f"SIZE MISMATCH {relpath}: {actual} != {expected}"
    print(f"  OK ({actual/1e9:.2f} GB)")
print("\nAll small files downloaded and verified.")


In [ ]:
# === Publish as dataset version ===
import subprocess, json, os

SEED = "/kaggle/working/seed"
DATASET_ID = "marsfmmg/viggle-animate-weights"
meta = {"title": "Viggle-Animate Model Weights", "id": DATASET_ID,
        "licenses": [{"name": "Apache-2.0"}]}
with open(os.path.join(SEED, "dataset-metadata.json"), "w") as f:
    json.dump(meta, f, indent=2)
print("Publishing (4GB, takes a few minutes)...")
r = subprocess.run(["kaggle", "datasets", "version", "-p", SEED,
    "-m", "Viggle small weights: LoRA + text_cond + VAE"],
    capture_output=True, text=True, timeout=3600)
print((r.stdout or "")[-300:])
assert r.returncode == 0, "Publish FAILED: " + (r.stderr or "")[-500:]
print("Done.")
